In [0]:
SELECT
    match_uid,
    match_players,
    game_mode,
    mvp_uid,
    svp_uid
FROM marvel_rivals.bronze.raw_match_details
LIMIT 3;


# Ticket SILV-001: Parse and Explode match_players
**Goal**: Write a SQL query that:
1. Reads from `marvel_rivals.bronze.raw_match_details`
2. Parses `match_players` froma raw JSON string into a proper array using `from_json()`
3. Explodes that array into one row per player
4. Selects `match_uid` plus the exploded player object

In [0]:
SELECT match_uid, EXPLODE(from_json(match_players, 'array<struct<
    player_uid: bigint,
    nick_name: string,
    camp: int,
    is_win: int,
    kills: int,
    deaths: int,
    assists: int,
    final_hits: int,
    solo_kill: int,
    total_hero_damage: double,
    total_hero_heal: double,
    total_damage_taken: double,
    player_heroes: string,
    dynamic_fields: string
>>')) AS player
FROM marvel_rivals.bronze.raw_match_details;

In [0]:
-- Check 1: how many distinct match uuids do we actually have?
SELECT COUNT(DISTINCT match_uid) AS unqiue_matches
FROM marvel_rivals.bronze.raw_match_details;

In [0]:
-- Check 2: Find matches with more or fewer than 12 players
SELECT match_uid, COUNT(*) as player_count
FROM (
    SELECT match_uid, EXPLODE(from_json(match_players, 'array<struct<
    player_uid: bigint,
    nick_name: string,
    camp: int,
    is_win: int,
    deaths: int,
    assists: int,
    final_hits: int,
    solo_kill: int,
    total_hero_damage: double,
    total_hero_heal: double,
    total_damage_taken: double,
    player_heroes: string,
    dynamic_fields: string
>>')) AS player
FROM marvel_rivals.bronze.raw_match_details
)
GROUP BY match_uid
HAVING COUNT(*) != 12
ORDER BY COUNT(*) DESC;

# Ticket SILV-002: Explode, Parse dynamic_fields, and Cap at 12 players
**Goal**: Build on SILV-001 to produce a clean, Silver-ready player-per-match dataset with:
1. One row per player per match (capped at 12 per match using `ROW_NUMBER()`)
2. `dynamic_fields` parsed out into individual columns (`add_score`, `new_score`, `level`, `system`)
3. `play_time` extracted from `player_heroes` (we'll fully explode heroes in the next ticket)

In [0]:
WITH exploded_players AS (
    SELECT 
        match_uid,
        player.player_uid,
        player.nick_name,
        player.camp,
        player.is_win,
        player.kills,
        player.deaths,
        player.assists,
        player.final_hits,
        player.solo_kill,
        player.total_hero_damage,
        player.total_hero_heal,
        player.total_damage_taken,
        player.player_heroes,
        player.dynamic_fields
    FROM (
        SELECT match_uid, EXPLODE(from_json(match_players, 'array<struct<
            player_uid: bigint,
            nick_name: string,
            camp: int,
            is_win: int,
            kills: int,
            deaths: int,
            assists: int,
            final_hits: int,
            solo_kill: int,
            total_hero_damage: double,
            total_hero_heal: double,
            total_damage_taken: double,
            player_heroes: string,
            dynamic_fields: string
        >>')) AS player
        FROM marvel_rivals.bronze.raw_match_details
    )
),
parsed_dynamic AS (
    SELECT
        match_uid,
        player_uid,
        nick_name,
        camp,
        is_win,
        kills,
        deaths,
        assists,
        final_hits,
        solo_kill,
        total_hero_damage,
        total_hero_heal,
        total_damage_taken,
        player_heroes,
        CAST(get_json_object(dynamic_fields, '$.name')      AS STRING) AS player_name,
        CAST(get_json_object(dynamic_fields, '$.level')     AS INT)    AS player_level,
        CAST(get_json_object(dynamic_fields, '$.new_level') AS INT)    AS new_level,
        CAST(get_json_object(dynamic_fields, '$.add_score') AS DOUBLE) AS add_score,
        CAST(get_json_object(dynamic_fields, '$.new_score') AS DOUBLE) AS new_score,
        CAST(get_json_object(dynamic_fields, '$.system')    AS STRING) AS system_console
    FROM exploded_players
),
ranked AS (
    SELECT
        ROW_NUMBER() OVER (
            PARTITION BY match_uid
            ORDER BY total_hero_damage + total_hero_heal + total_damage_taken DESC
        ) AS player_rank,
        *
    FROM parsed_dynamic
)
SELECT * EXCEPT (player_rank)
FROM ranked
WHERE player_rank <= 12

In [0]:
CREATE OR REPLACE TABLE marvel_rivals.silver.player_match_stats AS 
WITH exploded_players AS (
    SELECT 
        match_uid,
        player.player_uid,
        player.nick_name,
        player.camp,
        player.is_win,
        player.kills,
        player.deaths,
        player.assists,
        player.final_hits,
        player.solo_kill,
        player.total_hero_damage,
        player.total_hero_heal,
        player.total_damage_taken,
        player.player_heroes,
        player.dynamic_fields
    FROM (
        SELECT match_uid, EXPLODE(from_json(match_players, 'array<struct<
            player_uid: bigint,
            nick_name: string,
            camp: int,
            is_win: int,
            kills: int,
            deaths: int,
            assists: int,
            final_hits: int,
            solo_kill: int,
            total_hero_damage: double,
            total_hero_heal: double,
            total_damage_taken: double,
            player_heroes: string,
            dynamic_fields: string
        >>')) AS player
        FROM marvel_rivals.bronze.raw_match_details
    )
),
parsed_dynamic AS (
    SELECT
        match_uid,
        player_uid,
        nick_name,
        camp,
        is_win,
        kills,
        deaths,
        assists,
        final_hits,
        solo_kill,
        total_hero_damage,
        total_hero_heal,
        total_damage_taken,
        player_heroes,
        CAST(get_json_object(dynamic_fields, '$.name')      AS STRING) AS player_name,
        CAST(get_json_object(dynamic_fields, '$.level')     AS INT)    AS player_level,
        CAST(get_json_object(dynamic_fields, '$.new_level') AS INT)    AS new_level,
        CAST(get_json_object(dynamic_fields, '$.add_score') AS DOUBLE) AS add_score,
        CAST(get_json_object(dynamic_fields, '$.new_score') AS DOUBLE) AS new_score,
        CAST(get_json_object(dynamic_fields, '$.system')    AS STRING) AS system_console
    FROM exploded_players
),
ranked AS (
    SELECT
        ROW_NUMBER() OVER (
            PARTITION BY match_uid
            ORDER BY total_hero_damage + total_hero_heal + total_damage_taken DESC
        ) AS player_rank,
        *
    FROM parsed_dynamic
)
SELECT * EXCEPT (player_rank)
FROM ranked
WHERE player_rank <= 12

In [0]:
SELECT COUNT(*) as total_rows FROM marvel_rivals.silver.player_match_stats;
SELECT * FROM marvel_rivals.silver.player_match_stats LIMIT 3;

In [0]:
-- Average heroes played per player per match
SELECT 
    AVG(hero_count) as avg_heroes_per_player,
    MIN(hero_count) as min_heroes,
    MAX(hero_count) as max_heroes,
    COUNT(CASE WHEN hero_count = 1 THEN 1 END) as single_hero_players,
    COUNT(CASE WHEN hero_count > 1 THEN 1 END) as multi_hero_players
FROM (
    SELECT 
        match_uid,
        player_uid,
        SIZE(from_json(player_heroes, 'array<struct<
            hero_id: int,
            play_time: double,
            kills: int,
            deaths: int,
            assists: int,
            session_hit_rate: double,
            hero_icon: string
        >>')) as hero_count
    FROM marvel_rivals.silver.player_match_stats
)

In [0]:
WITH selected_heroes AS (
    SELECT
    match_uid,
    player_uid,
    nick_name, 
    camp,
    is_win,
    total_hero_damage,
    total_hero_heal,
    total_damage_taken,
    final_hits,
    solo_kill,
    add_score,
    new_score,
    player_level,
    new_level,
    system_console,
    player_name,
    hero.hero_id,
    hero.play_time,
    hero.kills,
    hero.deaths,
    hero.assists,
    hero.session_hit_rate,
    hero.hero_icon
    FROM (
        SELECT
        match_uid,
        player_uid,
        nick_name, 
        camp,
        is_win,
        total_hero_damage,
        total_hero_heal,
        total_damage_taken,
        final_hits,
        solo_kill,
        add_score,
        new_score,
        player_level,
        new_level,
        system_console,
        player_name,
        EXPLODE(from_json(player_heroes, 'array<struct<
            hero_id: int,
            play_time: double,
            kills: int,
            deaths: int,
            assists: int,
            session_hit_rate: double,
            hero_icon: string
        >>')) AS hero
        FROM marvel_rivals.silver.player_match_stats
    )
),
filtered_play_time AS (
    SELECT * FROM selected_heroes WHERE play_time >= 60
),
player_metrics AS (
    SELECT
        match_uid,
        player_uid,
        nick_name, 
        camp,
        is_win,
        final_hits,
        solo_kill,
        add_score,
        new_score,
        player_level,
        new_level,
        system_console,
        player_name,
        hero_id,
        play_time,
        session_hit_rate,
        ROUND((kills / play_time) * 600, 2) as kills_per_10_minutes,
        ROUND((deaths / play_time) * 600, 2) as deaths_per_10_minutes,
        ROUND((assists / play_time) * 600, 2) as assists_per_10_minutes,
        ROUND((total_hero_damage / play_time) * 600, 2) as damage_per_10_minutes,
        ROUND((total_hero_heal / play_time) * 600, 2) as healing_per_10_minutes,
        ROUND((total_damage_taken / play_time) * 600, 2) as damage_taken_per_10_minutes
    FROM filtered_play_time
)

In [0]:
CREATE OR REPLACE TABLE marvel_rivals.silver.player_hero_stats AS
WITH selected_heroes AS (
    SELECT
    match_uid,
    player_uid,
    nick_name, 
    camp,
    is_win,
    total_hero_damage,
    total_hero_heal,
    total_damage_taken,
    final_hits,
    solo_kill,
    add_score,
    new_score,
    player_level,
    new_level,
    system_console,
    player_name,
    hero.hero_id,
    hero.play_time,
    hero.kills,
    hero.deaths,
    hero.assists,
    hero.session_hit_rate,
    hero.hero_icon
    FROM (
        SELECT
        match_uid,
        player_uid,
        nick_name, 
        camp,
        is_win,
        total_hero_damage,
        total_hero_heal,
        total_damage_taken,
        final_hits,
        solo_kill,
        add_score,
        new_score,
        player_level,
        new_level,
        system_console,
        player_name,
        EXPLODE(from_json(player_heroes, 'array<struct<
            hero_id: int,
            play_time: double,
            kills: int,
            deaths: int,
            assists: int,
            session_hit_rate: double,
            hero_icon: string
        >>')) AS hero
        FROM marvel_rivals.silver.player_match_stats
    )
),
filtered_play_time AS (
    SELECT * FROM selected_heroes WHERE play_time >= 60
),
player_metrics AS (
    SELECT
        match_uid,
        player_uid,
        nick_name, 
        camp,
        is_win,
        final_hits,
        solo_kill,
        add_score,
        new_score,
        player_level,
        new_level,
        system_console,
        player_name,
        hero_id,
        play_time,
        session_hit_rate,
        ROUND((kills / play_time) * 600, 2) as kills_per_10_minutes,
        ROUND((deaths / play_time) * 600, 2) as deaths_per_10_minutes,
        ROUND((assists / play_time) * 600, 2) as assists_per_10_minutes,
        ROUND((total_hero_damage / play_time) * 600, 2) as damage_per_10_minutes,
        ROUND((total_hero_heal / play_time) * 600, 2) as healing_per_10_minutes,
        ROUND((total_damage_taken / play_time) * 600, 2) as damage_taken_per_10_minutes
    FROM filtered_play_time
)
SELECT *, current_timestamp() AS ingestion_timestamp
FROM player_metrics;

In [0]:
SELECT COUNT(*) as total_rows FROM marvel_rivals.silver.player_hero_stats;

In [0]:
-- Verify play_time filter worked (should return 0 rows)
SELECT COUNT(*) as short_play_rows
FROM marvel_rivals.silver.player_hero_stats
WHERE play_time < 60;

In [0]:
-- Check per-10 metrics look reasonable for a sample of known players
SELECT 
    nick_name,
    hero_id,
    play_time,
    kills_per_10_minutes,
    deaths_per_10_minutes,
    healing_per_10_minutes,
    damage_per_10_minutes,
    add_score
FROM marvel_rivals.silver.player_hero_stats
WHERE nick_name IN ('anonimis00', 'Boorex', 'Speeeeeeedwag')
LIMIT 10;

In [0]:
CREATE OR REPLACE TABLE marvel_rivals.silver.hero_lookup AS
SELECT
    CAST(id AS INT)          AS hero_id,
    name                     AS hero_name,
    real_name,
    role                     AS hero_role,
    attack_type,
    difficulty,
    CURRENT_TIMESTAMP()      AS ingestion_timestamp
FROM read_files(
    '/Volumes/marvel_rivals/bronze/raw_files/heroes.json',
    format => 'json',
    inferSchema => true
);

In [0]:
SELECT hero_id, hero_name, hero_role, attack_type 
FROM marvel_rivals.silver.hero_lookup
ORDER BY hero_id;

In [0]:
-- Which hero_ids in our match data have no lookup entry?
SELECT DISTINCT 
    phs.hero_id,
    hl.hero_name,
    hl.hero_role
FROM marvel_rivals.silver.player_hero_stats phs
LEFT JOIN marvel_rivals.silver.hero_lookup hl
    ON phs.hero_id = hl.hero_id
WHERE hl.hero_name IS NULL
ORDER BY phs.hero_id

In [0]:
CREATE OR REPLACE TABLE marvel_rivals.silver.hero_lookup AS
WITH combined AS (
    SELECT CAST(id AS INT) AS hero_id, name AS hero_name, real_name, 
           role AS hero_role, attack_type
    FROM read_files('/Volumes/marvel_rivals/bronze/raw_files/heroes.json',
        format => 'json', inferSchema => true)
    UNION ALL
    SELECT CAST(id AS INT) AS hero_id, name AS hero_name, real_name,
           role AS hero_role, attack_type
    FROM read_files('/Volumes/marvel_rivals/bronze/raw_files/missing_heroes.json',
        format => 'json', inferSchema => true)
),
deduplicated AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY hero_id ORDER BY hero_name) AS rn
    FROM combined
)
SELECT hero_id, hero_name, real_name, hero_role, attack_type,
       CURRENT_TIMESTAMP() AS ingestion_timestamp
FROM deduplicated
WHERE rn = 1

In [0]:
SELECT COUNT(*) as total_heroes FROM marvel_rivals.silver.hero_lookup;
SELECT hero_id, hero_name, hero_role FROM marvel_rivals.silver.hero_lookup ORDER BY hero_id;

In [0]:
CREATE OR REPLACE TABLE marvel_rivals.silver.player_hero_stats AS
WITH selected_heroes AS (
    SELECT
        match_uid,
        player_uid,
        nick_name, 
        camp,
        is_win,
        total_hero_damage,
        total_hero_heal,
        total_damage_taken,
        final_hits,
        solo_kill,
        add_score,
        new_score,
        player_level,
        new_level,
        system_console,
        player_name,
        hero.hero_id,
        hero.play_time,
        hero.kills,
        hero.deaths,
        hero.assists,
        hero.session_hit_rate,
        hero.hero_icon
    FROM (
        SELECT
            match_uid,
            player_uid,
            nick_name, 
            camp,
            is_win,
            total_hero_damage,
            total_hero_heal,
            total_damage_taken,
            final_hits,
            solo_kill,
            add_score,
            new_score,
            player_level,
            new_level,
            system_console,
            player_name,
            EXPLODE(from_json(player_heroes, 'array<struct<
                hero_id: int,
                play_time: double,
                kills: int,
                deaths: int,
                assists: int,
                session_hit_rate: double,
                hero_icon: string
            >>')) AS hero
        FROM marvel_rivals.silver.player_match_stats
    )
),
filtered_play_time AS (
    SELECT * FROM selected_heroes WHERE play_time >= 60
),
player_metrics AS (
    SELECT
        match_uid,
        player_uid,
        nick_name, 
        camp,
        is_win,
        final_hits,
        solo_kill,
        add_score,
        new_score,
        player_level,
        new_level,
        system_console,
        player_name,
        hero_id,
        play_time,
        session_hit_rate,
        ROUND((kills / play_time) * 600, 2)            AS kills_per_10_minutes,
        ROUND((deaths / play_time) * 600, 2)           AS deaths_per_10_minutes,
        ROUND((assists / play_time) * 600, 2)          AS assists_per_10_minutes,
        ROUND((total_hero_damage / play_time) * 600, 2) AS damage_per_10_minutes,
        ROUND((total_hero_heal / play_time) * 600, 2)   AS healing_per_10_minutes,
        ROUND((total_damage_taken / play_time) * 600, 2) AS damage_taken_per_10_minutes
    FROM filtered_play_time
)
SELECT 
    pm.*,
    hl.hero_name,
    hl.hero_role,
    hl.attack_type,
    CURRENT_TIMESTAMP() AS ingestion_timestamp
FROM player_metrics pm
LEFT JOIN marvel_rivals.silver.hero_lookup hl
    ON pm.hero_id = hl.hero_id

In [0]:
SELECT 
    hero_id,
    hero_name,
    hero_role,
    kills_per_10_minutes,
    healing_per_10_minutes,
    add_score
FROM marvel_rivals.silver.player_hero_stats
WHERE nick_name = 'anonimis00'
LIMIT 10

In [0]:
SELECT 
    hero_id,
    hero_name,
    hero_role,
    kills_per_10_minutes,
    healing_per_10_minutes,
    add_score
FROM marvel_rivals.silver.player_hero_stats
WHERE nick_name = 'Boorex'
LIMIT 10

In [0]:
SELECT * FROM marvel_rivals.bronze.raw_match_history LIMIT 2

In [0]:
CREATE OR REPLACE TABLE marvel_rivals.silver.squad_match_history AS 
SELECT 
    CAST(match_uid AS STRING) AS match_uid,
    CAST(match_season AS INT) AS match_season,
    CAST(match_map_id AS INT) AS match_map_id,
    CAST(match_play_duration AS DOUBLE) AS match_play_duration,
    CAST(match_time_stamp AS BIGINT) AS match_time_stamp,
    CAST(match_winner_side AS INT) match_winner_side,
    CAST(mvp_uid AS BIGINT) AS mvp_uid,
    CAST(svp_uid AS BIGINT) AS svp_uid,
    CAST(game_mode_id AS INT) AS game_mode_id,
    CAST(get_json_object(match_player, '$.player_uid') AS BIGINT) AS player_uid,
    CAST(get_json_object(match_player, '$.assists') AS INT) AS assists,
    CAST(get_json_object(match_player, '$.kills') AS INT) AS kills,
    CAST(get_json_object(match_player, '$.deaths') AS INT) AS deaths,
    CAST(get_json_object(match_player, '$.is_win.is_win') AS BOOLEAN) AS is_win,
    CAST(get_json_object(match_player, '$.disconnected') AS BOOLEAN) AS did_disconnect,
    CAST(get_json_object(match_player, '$.camp') AS INT) AS camp,
    CAST(get_json_object(match_player, '$.score_info.add_score') AS DOUBLE) AS add_score,
    CAST(get_json_object(match_player, '$.score_info.level') AS INT) AS previous_level,
    CAST(get_json_object(match_player, '$.score_info.new_level') AS INT) AS new_level,
    CAST(get_json_object(match_player, '$.score_info.new_score') AS DOUBLE) AS new_score,
    CAST(get_json_object(match_player, '$.player_hero.hero_id') AS INT) AS hero_id,
    CAST(get_json_object(match_player, '$.player_hero.hero_name') AS STRING) AS hero_name,
    CAST(get_json_object(match_player, '$.player_hero.total_hero_damage') AS DOUBLE) AS total_hero_damage,
    CAST(get_json_object(match_player, '$.player_hero.total_hero_heal') AS DOUBLE) AS total_hero_healing,
    CAST(get_json_object(match_player, '$.player_hero.total_damage_taken') AS DOUBLE) AS total_damage_taken,
    CAST(get_json_object(match_player, '$.player_hero.play_time') AS DOUBLE) AS play_time
    FROM marvel_rivals.bronze.raw_match_history;

In [0]:
SELECT COUNT(*) FROM marvel_rivals.silver.squad_match_history